This notebook is for all my actual model building and data preprocessing. My EDA is done in this other notebook: https://www.kaggle.com/code/richardhhong/calories-eda  
Also I know that a lot of my function and variable names suck but thats not important right now ~

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Feature Engineering
⭐ Features of BMI_to_weighttype, age_to_group, and combinations of categorical features taken from https://www.kaggle.com/code/kenangcmn/calories-xgboost-and-fe  
⭐ Feature of BMR from https://www.kaggle.com/code/ricopue/s5e5-calories-minute-great-target#Load-Python-Libraries 

In [3]:
from itertools import combinations

def bmi_to_weighttype(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi <= 24.9:
        return "NormalWeight"
    elif bmi <= 29.9:
        return "Overweight"
    else:
        return "Obesity"

def age_to_group(age):
    if age <= 18:
        return "Child"
    elif age <= 30:
        return "Young Adult"
    elif age <= 50:
        return "Adult"
    else:
        return "Senior"

def make_pairs(df):
    df_temp = df.copy()
    encode_columns = ['Sex', 'WeightType', 'AgeGroup']
    pair_size = [2,3]
    
    for r in pair_size:
        for cols in list(combinations(encode_columns, r)):
            new_col_name = '_'.join(cols)
            
            df_temp[new_col_name] = df_temp[list(cols)].astype(str).agg('_'.join, axis=1)
            df_temp[new_col_name] = df_temp[new_col_name].astype('category')

    return df_temp

In [4]:
def make_other_ints(df):
    df_temp = df.copy()
    df_temp['Intensity'] = df_temp['Heart_Rate'] / df_temp['Duration']
    return df_temp

def make_sex_int(df, features):
    df_temp = df.copy()
    for feature in features:
        df_temp[f'{feature}_Male'] = df_temp[feature] * df_temp['Sex_male']
        df_temp[f'{feature}_Female'] = df_temp[feature] * (1-df_temp['Sex_male'])
    return df_temp

def make_aggregates(df):
    df_temp = df.copy()
    numerical_cols = ['Age', "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]
    cat_cols = ['Sex_male']
    agg_types = ['min', 'max', 'mean']
    for cat_col in cat_cols:
        for agg_type in agg_types:
            aggs = df_temp[numerical_cols] - df_temp.groupby(cat_col)[numerical_cols].transform(agg_type)
            aggs.columns = [f"{cat_col}_{num_col}_{agg_type}" for num_col in aggs.columns]
            df_temp = pd.concat([df_temp, aggs], axis=1)
    return df_temp

def make_bmr(df):
    df_temp = df.copy()
    df_temp['BMR']=0
    df_temp.loc[df_temp.Sex_male==1,'BMR'] = df_temp['Weight'] * 9.65 + (df_temp['Height'] / 100) * 573 - df_temp['Age'] * 5.08 + 260
    df_temp.loc[df_temp.Sex_male==0,'BMR'] = df_temp['Weight'] * 7.38 + (df_temp['Height'] / 100) * 607 - df_temp['Age'] * 2.31 + 43
    df_temp.drop(columns=['Sex_male'], inplace=True)
    return df_temp

def make_features(df, test=False, make_int=True):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)
    
    # dummy encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'], drop_first=True)
    df_temp['Sex'] = df['Sex'].astype('category')
    
    df_temp = make_sex_int(df_temp, features=['Duration', 'Heart_Rate', 'Body_Temp', 'Age'])
    df_temp = make_aggregates(df_temp)

    df_temp['BMI'] = df_temp['Weight'] / (df_temp['Height']/100)**2
    df_temp["WeightType"] = df_temp["BMI"].apply(bmi_to_weighttype).astype("category")
    df_temp["AgeGroup"] = df_temp["Age"].apply(age_to_group).astype("category") 

    df_temp = make_pairs(df_temp)
    df_temp = make_bmr(df_temp)
    df_temp = make_other_ints(df_temp)

    return df_temp

# for predicting log of outcome rather than just outcome
def make_outcome_log(df):
    df_temp = df.copy()

    df_temp['log_calories'] = np.log1p(df_temp['Calories'])
    
    return df_temp

df_train1 = make_features(df_train, make_int = False)
df_train2 = make_outcome_log(df_train)

# Model

In [5]:
SEED = 30

In [6]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [7]:
# without the fancy features
X = df_train.drop(columns=['id', 'Calories'])
X['Sex'] = X['Sex'].astype('category')
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

# with the fancy features
X1 = df_train1.drop(columns=['Calories'])
y1 = df_train1['Calories']
X_train1, X_val1, _, _ = train_test_split(X1, y1, test_size=0.3, random_state=SEED)

# with log calories
y2 = df_train2['log_calories']
_, _, y_train2, y_val2 = train_test_split(X, y2, test_size=0.3, random_state=SEED)

In [8]:
# baseline without new features
cat_features=['Sex']
cat_baseline = CatBoostRegressor(cat_features = cat_features, verbose=0, random_seed=30)
cat_baseline.fit(X_train, y_train)

y_val_pred = cat_baseline.predict(X_val)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'CatBoost Baseline Score: {score}')

CatBoost Baseline Score: 0.062061827785241955


In [9]:
# baseline with new features
cat_features = list(X_train1.select_dtypes(include=['category']).columns)
cat_baseline1 = CatBoostRegressor(task_type="GPU", devices="0-2", cat_features=cat_features, verbose=0, random_seed=30)
cat_baseline1.fit(X_train1, y_train)

y_val_pred = cat_baseline1.predict(X_val1)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'Cat Baseline Score With new Features: {score}')

Cat Baseline Score With new Features: 0.06447980580151974


In [10]:
# baseline with log calories and the new features
cat_baseline3 = CatBoostRegressor(task_type="GPU", devices="0-2", cat_features=cat_features, verbose=0, random_seed=30)
cat_baseline3.fit(X_train1, y_train2)

y_val_pred = cat_baseline3.predict(X_val1)
y_val_pred = np.expm1(y_val_pred)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'Cat Baseline Score With new Features and Log Calories: {score}')

Cat Baseline Score With new Features and Log Calories: 0.060039836632127815


## Big Tuna

In [11]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [12]:
def rmsle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = np.sqrt(mean_squared_log_error(y_true, y_pred))
    return 'RMSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def cat_cv_rmsle(X, y, cat_features, params, num_folds=5, debug=False, log=False, y_act=y):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        y_val_act = y_act.iloc[val_idx]
        
        model = CatBoostRegressor(
            **params,
            cat_features=cat_features,
            verbose=0
        )
        model.fit(X_train,y_train)
        
        y_val_pred = model.predict(X_val)
        y_val_pred = np.maximum(0, y_val_pred)

        if log == True:
            y_val_pred = np.expm1(y_val_pred)
            
        score = np.sqrt(mean_squared_log_error(y_val_act, y_val_pred))

        if debug == True:
            print(score)
            
        fold_scores.append(score)
        
    return fold_scores

In [13]:
def objective(trial):
    params = {
        "task_type": "GPU",
        "devices": "0-2", 
        "iterations": trial.suggest_int("iterations", 1000, 3000, step=250),
        "depth": trial.suggest_int("depth", 3, 16),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "grow_policy": trial.suggest_categorical("grow_policy", ["Depthwise", "Lossguide", "SymmetricTree"]),
        "random_seed": SEED
    }
    feature_df = X1
    cat_features = list(feature_df.select_dtypes(include=['category']).columns)
    score = np.mean(cat_cv_rmsle(X=feature_df, 
                                 cat_features=cat_features,
                                 y=y2,
                                 params=params,
                                 debug=False,
                                 log=True))
    return score

In [14]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=20, show_progress_bar=True, )

In [15]:
# best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [16]:
best_params = {
    'iterations': 1500,
    'depth': 12,
    'learning_rate': 0.011540109964310583,
    'l2_leaf_reg': 8.201103202153114,
    'border_count': 226,
    'random_strength': 1.322486803055711,
    'bagging_temperature': 0.20648364149092824,
    'grow_policy': 'SymmetricTree'
}

# Submission

In [17]:
cat_features = list(X1.select_dtypes(include=['category']).columns)
best_model = CatBoostRegressor(**best_params,
                               task_type= "GPU",
                               devices="0-2",
                               cat_features=cat_features,
                               verbose=0)
best_model.fit(X_train1, y_train2)

In [18]:
y_val_pred = best_model.predict(X_val1)
y_val_pred = np.expm1(y_val_pred)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(score)

0.059961332497834405


In [19]:
df_test1 = make_features(df_test, test=True)

df_test1.head()

,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Sex,Duration_Male,Duration_Female,Heart_Rate_Male,...,Sex_male_Body_Temp_mean,BMI,WeightType,AgeGroup,Sex_WeightType,Sex_AgeGroup,WeightType_AgeGroup,Sex_WeightType_AgeGroup,BMR,Intensity
0,45,177.0,81.0,7.0,87.0,39.8,male,7.0,0.0,87.0,...,-0.222585,25.854639,Overweight,Adult,male_Overweight,male_Adult,Overweight_Adult,male_Overweight_Adult,1827.26,12.428571
1,26,200.0,97.0,20.0,101.0,40.5,male,20.0,0.0,101.0,...,0.477415,24.250000,NormalWeight,Young Adult,male_NormalWeight,male_Young Adult,NormalWeight_Young Adult,male_NormalWeight_Young Adult,2209.97,5.050000
2,29,188.0,85.0,16.0,102.0,40.4,female,0.0,16.0,0.0,...,0.350460,24.049344,NormalWeight,Young Adult,female_NormalWeight,female_Young Adult,NormalWeight_Young Adult,female_NormalWeight_Young Adult,1744.47,6.375000
3,39,172.0,73.0,20.0,107.0,40.6,female,0.0,20.0,0.0,...,0.550460,24.675500,NormalWeight,Adult,female_NormalWeight,female_Adult,NormalWeight_Adult,female_NormalWeight_Adult,1535.69,5.350000
4,30,173.0,67.0,16.0,94.0,40.5,female,0.0,16.0,0.0,...,0.450460,22.386314,NormalWeight,Young Adult,female_NormalWeight,female_Young Adult,NormalWeight_Young Adult,female_NormalWeight_Young Adult,1518.27,5.875000


In [20]:
y_test_pred = best_model.predict(df_test1)
y_test_pred = np.expm1(y_test_pred)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,27.518348
1,750001,108.263434
2,750002,86.991072
3,750003,124.681434
4,750004,75.760855
